In [1]:
# cell 1 — install dependencies
# scikit-learn not pre-installed on kaggle, everything else is
!pip install scikit-learn --quiet

In [2]:
# cell 2 — configuration + auto-detect dataset paths
# edit MODELS_TO_RUN if you need to resume a partial session

import os

# auto-detect Training/ and Testing/ regardless of how kaggle names the input folder
# this handles cases where the dataset has an extra nested directory level
def find_dataset_dirs(input_root='/kaggle/input'):
    train_dir = test_dir = None
    for root, dirs, files in os.walk(input_root):
        if 'Training' in dirs and train_dir is None:
            candidate = os.path.join(root, 'Training')
            subdirs = [d for d in os.listdir(candidate) if os.path.isdir(os.path.join(candidate, d))]
            if len(subdirs) >= 4:  # expect 4 class folders
                train_dir = candidate
        if 'Testing' in dirs and test_dir is None:
            candidate = os.path.join(root, 'Testing')
            subdirs = [d for d in os.listdir(candidate) if os.path.isdir(os.path.join(candidate, d))]
            if len(subdirs) >= 4:
                test_dir = candidate
        if train_dir and test_dir:
            break
    return train_dir, test_dir

TRAIN_DIR, TEST_DIR = find_dataset_dirs()

if TRAIN_DIR is None or TEST_DIR is None:
    # print the actual input structure so we can diagnose the issue
    print('ERROR: could not find Training/ or Testing/ folders.')
    print('actual contents of /kaggle/input:')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth > 3:
            continue
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
    raise RuntimeError(
        'dataset not found — make sure you added the brain tumor MRI dataset '
        'in step 4 of the instructions, then restart and run all cells again'
    )

WORK_DIR  = '/kaggle/working'
RUNS_DIR  = os.path.join(WORK_DIR, 'runs')

# training config (matches blueprint exactly — do not change)
SEEDS        = [42, 1337, 2026]
MAX_EPOCHS   = 30
PATIENCE     = 7
BATCH_SIZE   = 32
NUM_WORKERS  = 2

# models to run — remove any you have already completed
MODELS_TO_RUN = ['resnet50', 'efficientnet_b0', 'vit_b16', 'hybrid']

# create output directories
for model_name in MODELS_TO_RUN:
    for seed in SEEDS:
        os.makedirs(os.path.join(RUNS_DIR, model_name, f'seed_{seed}'), exist_ok=True)

print('config ok')
print(f'  train dir : {TRAIN_DIR}')
print(f'  test dir  : {TEST_DIR}')
print(f'  output    : {RUNS_DIR}')
print(f'  models    : {MODELS_TO_RUN}')
print(f'  seeds     : {SEEDS}')

config ok
  train dir : /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training
  test dir  : /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing
  output    : /kaggle/working/runs
  models    : ['resnet50', 'efficientnet_b0', 'vit_b16', 'hybrid']
  seeds     : [42, 1337, 2026]


In [3]:
# cell 3 — create stratified train/val split (85/15, seed=42)
import json
from sklearn.model_selection import train_test_split

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

all_files, all_labels = [], []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    for fname in sorted(os.listdir(cls_dir)):
        if os.path.isfile(os.path.join(cls_dir, fname)):  # skip hidden files
            all_files.append(f'{cls}/{fname}')
            all_labels.append(cls)

train_files, val_files, train_labels, val_labels = train_test_split(
    all_files, all_labels, test_size=0.15, random_state=42, stratify=all_labels
)

split = {
    'seed': 42, 'val_ratio': 0.15,
    'train_count': len(train_files), 'val_count': len(val_files),
    'train': sorted(train_files), 'val': sorted(val_files),
}
SPLIT_FILE = os.path.join(WORK_DIR, 'train_val_split.json')
with open(SPLIT_FILE, 'w') as f:
    json.dump(split, f)

print(f'split created: train={len(train_files)}  val={len(val_files)}')

split created: train=4760  val=840


In [4]:
# cell 4 — dataset and dataloader definitions
import json
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

CLASS_TO_IDX  = {c: i for i, c in enumerate(CLASS_NAMES)}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transforms():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def get_val_test_transforms():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

class KaggleTumorDataset(Dataset):
    def __init__(self, train_dir, split_file, subset, transform=None):
        with open(split_file) as f:
            split = json.load(f)
        self.transform = transform
        self.samples = [
            (os.path.join(train_dir, p.replace('/', os.sep)), CLASS_TO_IDX[p.split('/')[0]])
            for p in split[subset]
        ]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

class KaggleTestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.transform = transform
        self.samples = [
            (os.path.join(test_dir, cls, fname), CLASS_TO_IDX[cls])
            for cls in CLASS_NAMES
            for fname in sorted(os.listdir(os.path.join(test_dir, cls)))
            if os.path.isfile(os.path.join(test_dir, cls, fname))
        ]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

def get_dataloaders(batch_size=32, num_workers=2):
    import torch
    pin = torch.cuda.is_available()
    train_ds = KaggleTumorDataset(TRAIN_DIR, SPLIT_FILE, 'train', get_train_transforms())
    val_ds   = KaggleTumorDataset(TRAIN_DIR, SPLIT_FILE, 'val',   get_val_test_transforms())
    test_ds  = KaggleTestDataset(TEST_DIR, get_val_test_transforms())
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=pin),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin),
    )

print('dataset classes defined')

dataset classes defined


In [5]:
# cell 5 — model definitions
import torch
import torch.nn as nn
from torchvision import models

def build_resnet50(num_classes=4, pretrained=True):
    weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.resnet50(weights=weights)
    m.fc = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(2048, num_classes))
    return m

def build_efficientnet_b0(num_classes=4, pretrained=True):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.efficientnet_b0(weights=weights)
    m.classifier = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(1280, num_classes))
    return m

def build_vit_b16(num_classes=4, pretrained=True):
    weights = models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.vit_b_16(weights=weights)
    m.heads.head = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(768, num_classes))
    return m

class HybridCNNViT(nn.Module):
    def __init__(self, num_classes=4, pretrained=True):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        bb = models.resnet50(weights=weights)
        self.cnn = nn.Sequential(
            bb.conv1, bb.bn1, bb.relu, bb.maxpool,
            bb.layer1, bb.layer2, bb.layer3, bb.layer4
        )
        self.proj      = nn.Linear(2048, 512)
        self.pos_embed = nn.Parameter(torch.randn(1, 49, 512) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, 512) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=512, nhead=8, dim_feedforward=2048, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=4)
        self.head = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(512, num_classes))

    def forward(self, x):
        b = x.size(0)
        feat   = self.cnn(x).flatten(2).transpose(1, 2)   # (b,49,2048)
        tokens = self.proj(feat) + self.pos_embed          # (b,49,512)
        cls    = self.cls_token.expand(b, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)           # (b,50,512)
        out    = self.transformer(tokens)                  # (b,50,512)
        return self.head(out[:, 0])

def build_model(name, num_classes=4, pretrained=True):
    builders = {
        'resnet50':        build_resnet50,
        'efficientnet_b0': build_efficientnet_b0,
        'vit_b16':         build_vit_b16,
        'hybrid':          lambda nc, pt: HybridCNNViT(nc, pt),
    }
    return builders[name](num_classes, pretrained)

print('model definitions loaded')

model definitions loaded


In [6]:
# cell 6 — utilities
import random
import numpy as np
import torch
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def compute_metrics(targets, preds):
    return {
        'accuracy':        accuracy_score(targets, preds),
        'macro_f1':        f1_score(targets, preds, average='macro', zero_division=0),
        'macro_precision': precision_score(targets, preds, average='macro', zero_division=0),
        'macro_recall':    recall_score(targets, preds, average='macro', zero_division=0),
    }

def save_checkpoint(state, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(state, path)

def save_epoch_log(log_path, records):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, 'w') as f:
        json.dump(records, f, indent=2)

print('utilities loaded')

utilities loaded


In [7]:
# cell 7 — training loop
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm

def run_epoch(model, loader, criterion, optimizer, device, training):
    model.train() if training else model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            if training: optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_targets.extend(labels.cpu().tolist())
    return total_loss / len(loader.dataset), compute_metrics(all_targets, all_preds)

def train_one_run(model_name, seed):
    """train one model with one seed. saves checkpoint and log."""
    run_dir   = os.path.join(RUNS_DIR, model_name, f'seed_{seed}')
    ckpt_path = os.path.join(run_dir, 'best.pt')
    log_path  = os.path.join(run_dir, 'log.json')

    # skip if already done
    if os.path.isfile(ckpt_path) and os.path.isfile(log_path):
        print(f'  [{model_name} seed={seed}] already done — skipping')
        return

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_seed(seed)

    train_loader, val_loader, _ = get_dataloaders(BATCH_SIZE, NUM_WORKERS)
    model     = build_model(model_name, pretrained=True).to(device)
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    best_f1, best_epoch, patience_count = 0.0, 0, 0
    records = []

    print(f'\n[{model_name}] seed={seed}  device={device}')
    print('-' * 70)

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_m = run_epoch(model, train_loader, criterion, optimizer, device, True)
        val_loss,   val_m   = run_epoch(model, val_loader,   criterion, optimizer, device, False)
        scheduler.step()

        val_f1 = val_m['macro_f1']
        record = {
            'epoch': epoch,
            'train_loss': round(train_loss, 6), 'train_f1': round(train_m['macro_f1'], 6),
            'train_accuracy': round(train_m['accuracy'], 6),
            'val_loss': round(val_loss, 6),     'val_f1': round(val_f1, 6),
            'val_accuracy': round(val_m['accuracy'], 6),
            'lr': round(scheduler.get_last_lr()[0], 8),
        }
        records.append(record)

        print(
            f'  epoch {epoch:02d}/{MAX_EPOCHS} | '
            f'train_loss={train_loss:.4f} train_f1={train_m["macro_f1"]:.4f} | '
            f'val_loss={val_loss:.4f} val_f1={val_f1:.4f} | '
            f'lr={scheduler.get_last_lr()[0]:.2e}'
        )

        if val_f1 > best_f1:
            best_f1, best_epoch, patience_count = val_f1, epoch, 0
            save_checkpoint({
                'model_name': model_name, 'seed': seed, 'epoch': epoch,
                'best_f1': best_f1, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
            }, ckpt_path)
            print(f'    *** new best — checkpoint saved (val_f1={best_f1:.4f}) ***')
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f'\n  early stopping at epoch {epoch} (patience={PATIENCE})')
                break

    save_epoch_log(log_path, records)
    print(f'  done. best epoch={best_epoch}  best val_f1={best_f1:.4f}')
    print(f'  log saved to {log_path}')

print('training loop defined')

training loop defined


In [8]:
# cell 8 — verify dataset and gpu before training
import torch

print('verifying dataset...')
for split_name, split_dir in [('Training', TRAIN_DIR), ('Testing', TEST_DIR)]:
    total = 0
    for cls in CLASS_NAMES:
        cls_path = os.path.join(split_dir, cls)
        n = len([f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))])
        total += n
        print(f'  {split_name}/{cls}: {n}')
    print(f'  {split_name} total: {total}')
    print()

print(f'cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: no gpu — enable GPU in settings before continuing')

verifying dataset...
  Training/glioma: 1400
  Training/meningioma: 1400
  Training/notumor: 1400
  Training/pituitary: 1400
  Training total: 5600

  Testing/glioma: 400
  Testing/meningioma: 400
  Testing/notumor: 400
  Testing/pituitary: 400
  Testing total: 1600

cuda available: True
gpu: Tesla T4


In [9]:
# cell 9 — run all 12 training runs
# ~5-6 hours on T4 GPU
# already-completed runs are skipped automatically

import time
start_total = time.time()

for model_name in MODELS_TO_RUN:
    for seed in SEEDS:
        t0 = time.time()
        train_one_run(model_name, seed)
        elapsed = (time.time() - t0) / 60
        print(f'  [{model_name} seed={seed}] elapsed: {elapsed:.1f} min')

total_elapsed = (time.time() - start_total) / 60
print(f'\nall training runs complete. total time: {total_elapsed:.1f} min')

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 205MB/s] 



[resnet50] seed=42  device=cuda
----------------------------------------------------------------------


  0%|          | 0/149 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

...

  0%|          | 0/27 [00:00<?, ?it/s]

  epoch 10/30 | train_loss=0.2320 train_f1=0.9903 | val_loss=0.2642 val_f1=0.9796 | lr=7.50e-05


  0%|          | 0/149 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

  epoch 11/30 | train_loss=0.2221 train_f1=0.9926 | val_loss=0.2349 val_f1=0.9869 | lr=7.03e-05

  early stopping at epoch 11 (patience=7)
  done. best epoch=4  best val_f1=0.9905
  log saved to /kaggle/working/runs/hybrid/seed_2026/log.json
  [hybrid seed=2026] elapsed: 13.4 min

all training runs complete. total time: 462.5 min


In [10]:
# cell 10 — package all results and create download link
#
# creates benchmark_results_full.zip with all checkpoints and logs
#
# after downloading, place the zip at:
#   C:\Users\mahes\Desktop\STUFF\brain-tumor-bench\results\kaggle_runs\benchmark_results_full.zip
#
import zipfile
from IPython.display import FileLink, display

full_zip = os.path.join(WORK_DIR, 'benchmark_results_full.zip')

with zipfile.ZipFile(full_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(RUNS_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname   = os.path.relpath(full_path, RUNS_DIR)
            zipf.write(full_path, arcname)

size_mb = os.path.getsize(full_zip) / (1024 * 1024)
print(f'archive created: {full_zip}')
print(f'size: {size_mb:.1f} MB')
print()
print('download the file below, then place it at:')
print('  C:\\Users\\mahes\\Desktop\\STUFF\\brain-tumor-bench\\results\\kaggle_runs\\benchmark_results_full.zip')
print()
display(FileLink('benchmark_results_full.zip'))

archive created: /kaggle/working/benchmark_results_full.zip
size: 4791.4 MB

download the file below, then place it at:
  C:\Users\mahes\Desktop\STUFF\brain-tumor-bench\results\kaggle_runs\benchmark_results_full.zip



/kaggle/working/benchmark_results_full.zip